[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tuesdaythe13th/multilingualcompositionalsafety_evals/blob/main/fairness_failure_dashboard.ipynb)

# ARTIFEX — Cohort Fairness & Failure Cluster Dashboard
### `fairness_failure_dashboard.ipynb`

```
 ███████╗ █████╗ ██╗██████╗ ███╗   ██╗███████╗███████╗███████╗
 ██╔════╝██╔══██╗██║██╔══██╗████╗  ██║██╔════╝██╔════╝██╔════╝
 █████╗  ███████║██║██████╔╝██╔██╗ ██║█████╗  ███████╗███████╗
 ██╔══╝  ██╔══██║██║██╔══██╗██║╚██╗██║██╔══╝  ╚════██║╚════██║
 ██║     ██║  ██║██║██║  ██║██║ ╚████║███████╗███████║███████║
 ╚═╝     ╚═╝  ╚═╝╚═╝╚═╝  ╚═╝╚═╝  ╚═══╝╚══════╝╚══════╝╚══════╝
```

**Goal**: Load `results/*_metrics.json` files, visualize per-language and per-locale
performance gaps (micro_f1), and export a structured failure cluster manifest.

**Schema**: Reads files produced by the ARTIFEX evaluator pipeline.
Each `*_metrics.json` contains:
```json
{
  "slice": "<eval_name>",
  "n": 120,
  "micro_f1": 0.74,
  "by_language": {"es-MX": 0.68, "es-ES": 0.81},
  "by_locale":   {"CO": 0.61, "ES": 0.83}
}
```

**Principal Engineer**: Tuesday · ARTIFEX Labs  
tuesday@artifex.fun · linktr.ee/artifexlabs · huggingface.co/222tuesday  
contact: zcal.co/tuesday

In [ ]:
#@title ① Load results/*_metrics.json
import json
import glob
import os
from pathlib import Path
import pandas as pd

RESULTS_GLOB = 'results/*_metrics.json'

# Allow override via env (useful for mounted Drive)
results_glob = os.environ.get('ARTIFEX_RESULTS_GLOB', RESULTS_GLOB)
files = glob.glob(results_glob)

if not files:
    # Synthesize demo data so the notebook is runnable without real results
    print('⚠️  No results found at:', results_glob)
    print('   Generating synthetic demo data...')
    os.makedirs('results', exist_ok=True)
    demo_payloads = [
        {'slice': 'spanish_benchmark',    'n': 68,  'micro_f1': 0.74,
         'by_language': {'es-MX': 0.68, 'es-ES': 0.81, 'es-CO': 0.65},
         'by_locale':   {'MX': 0.68, 'ES': 0.83, 'CO': 0.61, 'AR': 0.72}},
        {'slice': 'dialect_divergence',   'n': 200, 'micro_f1': 0.71,
         'by_language': {'es-MX': 0.67, 'es-ES': 0.77},
         'by_locale':   {'MX': 0.66, 'ES': 0.79}},
        {'slice': 'ailuminate_jailbreak', 'n': 500, 'micro_f1': 0.82,
         'by_language': {'es-MX': 0.79, 'es-AR': 0.80, 'es-ES': 0.88, 'es-CO': 0.77},
         'by_locale':   {'MX': 0.79, 'AR': 0.80, 'ES': 0.88, 'CO': 0.77}},
    ]
    for payload in demo_payloads:
        path = f'results/{payload["slice"]}_metrics.json'
        with open(path, 'w') as f:
            json.dump(payload, f, indent=2)
    files = glob.glob(results_glob)

records = []
for fpath in sorted(files):
    with open(fpath) as f:
        data = json.load(f)
    print(f'  {Path(fpath).name:45s}  slice={data.get("slice","?")}  n={data.get("n","?")}  micro_f1={data.get("micro_f1","?")}' )
    records.append(data)

print(f'\n✅ Loaded {len(records)} results file(s).')

In [ ]:
#@title ② Slice Summarizer — by_language & by_locale Gaps
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

def summarize_slice(record: dict) -> None:
    """Print and plot micro_f1 gaps for a single results record."""
    slice_name = record.get('slice', 'unknown')
    n          = record.get('n', '?')
    overall    = record.get('micro_f1', None)

    print(f'\n══ Slice: {slice_name} (n={n})  overall micro_f1={overall} ══')

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f'{slice_name} — Performance Gaps', fontsize=12, fontweight='bold')

    for ax, key, label in [
        (axes[0], 'by_language', 'Language Code'),
        (axes[1], 'by_locale',   'Locale'),
    ]:
        breakdown = record.get(key, {})
        if not breakdown:
            ax.text(0.5, 0.5, f'No {key} data', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(label)
            continue

        sorted_items = sorted(breakdown.items(), key=lambda x: x[1])
        labels, values = zip(*sorted_items)
        colors = ['#d7191c' if v < 0.7 else '#fdae61' if v < 0.8 else '#1a9641' for v in values]

        ax.barh(labels, values, color=colors)
        if overall is not None:
            ax.axvline(overall, color='black', linestyle='--', linewidth=1.2, label=f'overall={overall}')
            ax.legend(fontsize=8)
        ax.set_xlim(0, 1.05)
        ax.set_xlabel('micro_f1')
        ax.set_title(f'{label} Breakdown')
        ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))

        # Print sorted table
        print(f'  [{key}]')
        for lbl, val in sorted_items:
            gap = f'  Δ{val - overall:+.3f}' if overall is not None else ''
            flag = ' ⚠️' if val < 0.7 else ''
            print(f'    {lbl:10s}  {val:.4f}{gap}{flag}')

    plt.tight_layout()
    plt.savefig(f'fairness_{slice_name}.png', dpi=120, bbox_inches='tight')
    plt.show()

for rec in records:
    summarize_slice(rec)

In [ ]:
#@title ③ Cross-Slice Language Heatmap
import numpy as np
import seaborn as sns

# Build language × slice matrix
all_langs = sorted({lang for r in records for lang in r.get('by_language', {}).keys()})
slice_names = [r.get('slice', f'slice_{i}') for i, r in enumerate(records)]

if all_langs and len(records) > 1:
    matrix = np.full((len(all_langs), len(records)), np.nan)
    for j, rec in enumerate(records):
        by_lang = rec.get('by_language', {})
        for i, lang in enumerate(all_langs):
            if lang in by_lang:
                matrix[i, j] = by_lang[lang]

    fig, ax = plt.subplots(figsize=(max(6, len(records) * 2), max(4, len(all_langs) * 0.7)))
    mask = np.isnan(matrix)
    sns.heatmap(
        matrix, mask=mask,
        annot=True, fmt='.2f',
        xticklabels=slice_names,
        yticklabels=all_langs,
        cmap='RdYlGn', vmin=0.5, vmax=1.0,
        linewidths=0.5,
        ax=ax
    )
    ax.set_title('micro_f1 by Language × Evaluation Slice', fontsize=12, fontweight='bold')
    ax.set_xlabel('Evaluation Slice')
    ax.set_ylabel('Language Code')
    plt.tight_layout()
    plt.savefig('fairness_heatmap.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('✅ Heatmap saved to fairness_heatmap.png')
else:
    print('Only one slice or no language breakdowns — skipping cross-slice heatmap.')

In [ ]:
#@title ④ Failure Cluster Manifest — Schema & Export
# This cell documents the failure cluster schema and generates a starter manifest.
# Human analysts should fill in the free-text fields (failure_pattern, recommended_fix).

import json
from datetime import datetime, timezone

FAILURE_CLUSTER_SCHEMA = {
    '_schema_version': '1.0',
    '_description': 'ARTIFEX Failure Cluster Manifest — one record per identified error cluster',
    '_fields': {
        'cluster_id':        'str  — unique ID, e.g. FC-001',
        'slice':             'str  — eval slice name matching *_metrics.json',
        'risk_category':     'str  — CBRN / CSAM / Harassment / Deception / Other',
        'modality_mix':      'str  — text / text+image / audio',
        'language':          'str  — BCP-47 language code',
        'locale':            'str  — ISO 3166-1 alpha-2 country code',
        'failure_pattern':   'str  — free-text description of the recurring error pattern',
        'prevalence':        'float — fraction of slice items exhibiting this pattern',
        'severity_impact':   'str  — HIGH / MEDIUM / LOW',
        'recommended_fix':   'str  — free-text mitigation / data collection recommendation',
    }
}

# Auto-generate starter entries for slices with low-scoring languages
starter_clusters = []
fc_id = 1
for rec in records:
    for lang, f1 in rec.get('by_language', {}).items():
        if f1 < 0.70:
            starter_clusters.append({
                'cluster_id':       f'FC-{fc_id:03d}',
                'slice':            rec.get('slice', ''),
                'risk_category':    'TBD',
                'modality_mix':     'text',
                'language':         lang,
                'locale':           lang.split('-')[-1] if '-' in lang else '',
                'failure_pattern':  f'Auto-flagged: micro_f1={f1:.3f} < 0.70 threshold',
                'prevalence':       round(1 - f1, 3),
                'severity_impact':  'HIGH' if f1 < 0.60 else 'MEDIUM',
                'recommended_fix':  'Collect additional annotated examples for this language/locale.',
            })
            fc_id += 1

manifest = {
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'schema':        FAILURE_CLUSTER_SCHEMA,
    'clusters':      starter_clusters,
}

out_path = 'failure_cluster_manifest.json'
with open(out_path, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f'✅ Failure cluster manifest written to {out_path}')
print(f'   {len(starter_clusters)} auto-flagged cluster(s) (micro_f1 < 0.70)')

# Pretty-print manifest
if starter_clusters:
    import pandas as pd
    display(pd.DataFrame(starter_clusters)[[
        'cluster_id','slice','language','severity_impact','prevalence','failure_pattern'
    ]])
else:
    print('   No language slices below the 0.70 threshold — excellent fairness coverage!')